# ESOL exploration

ESOL (Delaney): 1,128 compounds with measured aqueous solubility (logS, mol/L).
Build the mmpdb database first with `just esol`. All mmpdb access goes through `moleculedb.mmp`.

In [ ]:
import pandas as pd
import networkx as nx
from rdkit import Chem
from rdkit.Chem import Draw

from moleculedb import esol

df = pd.read_csv(esol.CSV_PATH)
df = df.rename(columns={"measured log solubility in mols per litre": "logS", "Compound ID": "name"})
df.head()

In [ ]:
df.describe()

## Most and least soluble

In [ ]:
def grid(rows):
    mols = [Chem.MolFromSmiles(s) for s in rows.smiles]
    legends = [f"{n}\nlogS {v:.2f}" for n, v in zip(rows.name, rows.logS)]
    return Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(220, 180), legends=legends)

grid(df.nlargest(8, "logS"))

## Matched molecular pairs

`db.neighbors(id)` lists compounds one fragment swap away; `db.graph()` gives the whole thing as a networkx graph.

In [ ]:
db = esol.database()
names = esol.names()
logS = {cid: c.properties["logS"] for cid, c in db.compounds().items()}

cid = next(k for k, v in names.items() if v == "Chlorobenzene")
pd.DataFrame(
    {"to": names[p.target], "transform": f"{p.from_smiles} >> {p.to_smiles}", "delta_logS": logS[p.target] - logS[cid], "size": p.size}
    for p in db.neighbors(cid, max_size=3)
).sort_values("delta_logS", ascending=False)

In [ ]:
g = db.graph(max_size=2)
components = sorted(nx.connected_components(g), key=len, reverse=True)
print(g.number_of_nodes(), "compounds,", g.number_of_edges(), "pairs,", sum(len(c) > 1 for c in components), "clusters")
print("largest clusters:", [len(c) for c in components[:10]])